# 25. W8A16 Quantization | W8A16 量化
**难度：** Medium | **环境：** CPU-first | **标签：** `量化压缩`, `W8A16`, `Linear` | **目标人群：** 量化压缩学习者

---

## 本节导读

模型变大以后，权重会占用大量显存，前向计算也要反复读取这些权重。量化先从最容易观察的对象开始：只改变权重的存储表示，看看显存账本和输出误差会怎样变化。

本节从一个线性变换出发，跟踪“浮点权重 → 8-bit 表示 → 前向计算”的过程。学习时需要回答三个问题：权重为什么能压缩、激活为什么仍保持较高精度、反量化后输出与原始计算相差多少。

学完本节后，你可以把 W8A16 与后续的 4-bit / QLoRA、GPTQ / AWQ 和真实 backend 测量联系起来。

**关键词：** `W8A16`, `INT8`, `quantization`

---


## 前置阅读

**导语：** 先认识 dtype、量化对象和误差，再观察权重存储精度变化如何影响前向计算。
- [核心前置：P1 · 01 Data Types and Precision | 数据类型与精度](../01_Hardware_Math_and_Systems/01_Data_Types_and_Precision.ipynb)
- [核心前置：P1 · 21 Quantization Theory and INT4/INT8 | 量化理论与 INT4/INT8](../01_Hardware_Math_and_Systems/21_Quantization_Theory_and_INT4_INT8.ipynb)
- [可选扩展：P1 · 12 TensorCore and Mixed Precision | Tensor Core 与混合精度](../01_Hardware_Math_and_Systems/12_TensorCore_and_Mixed_Precision.ipynb)

---


### Step 1: 量化对象、时机与 W8A16 总览

量化首先改变的是模型状态的表示方式：本节把对象固定为权重，把激活保持在 FP16/FP32，并沿着“浮点权重 → INT8 与 scale → 反量化前向”的链路观察存储和误差。`W8` 表示权重以 8-bit 形式保存，`A16` 表示激活仍沿用 16-bit（教学实现也允许 FP32）路径。主图把量化对象、处理阶段和 W8A16 前向路径放在同一张图中，便于比较存储表示、计算表示和最终输出。

| 本节对象 | 输入 | 机制 | 输出与观察指标 |
|---|---|---|---|
| 浮点权重 | FP32 / FP16 权重张量 | absmax 对称量化 | `torch.int8` 权重、scale、存储字节数 |
| W8A16 线性层 | INT8 权重 + FP16/FP32 激活 | 前向前反量化 | 输出形状、MSE / 余弦相似度 |
| 机制扩展与部署 | 校准权重、运行时状态和目标 backend | GPTQ/AWQ、FP8、KV Cache 量化各自改变不同成本 | 后续再验证完整模型显存、延迟、吞吐和质量 |

![W8A16 量化流程图](../docs/public/02_PyTorch_Algorithms/25_quantization_pipeline_cn.svg)


### Step 2: 量化数据流如何改变存储与计算表示

本步先把量化看成一条数据流：高精度权重根据动态范围映射到整数表示，同时保存缩放信息；前向计算时，再让低比特权重和高精度激活共同产生输出。重点是分清每一步保存了什么、改变了什么，以及误差从哪里进入。

| 流程阶段 | 主要变化 | 保留的信息 | 观察重点 |
|---|---|---|---|
| 输入 | 高精度权重进入量化流程 | FP16 / FP32 权重 | 作为误差参照 |
| 映射 | 根据动态范围确定缩放关系，并映射到有限整数范围 | 缩放信息、整数权重 | 是否覆盖原始数值范围 |
| 存储 | 权重以低比特形式保存，激活仍保持较高精度 | INT8 权重、scale、FP16 / FP32 激活 | 权重存储与计算精度分开 |
| 前向 | 反量化参与矩阵计算并产生近似输出 | 输出张量 | 形状、误差与计算代价 |


### Step 3: 如何由动态范围得到量化与反量化结果

W8A16 先根据动态范围确定 scale，再完成整数映射和恢复。这里的公式解释了 Step 2 中“映射”和“前向”两个阶段如何衔接。采用以 127 为正向上限的对称教学实现，整数结果会被限制在 INT8 可表示范围内。比如权重 `x=2.5` 且 `absmax=2.5` 时，`scale=127/2.5=50.8`，量化值约为 `round(2.5×50.8)=127`，恢复后再得到接近 `2.5` 的浮点值。量化误差来自舍入和截断，不等于完整模型质量损失。

| 阶段 | 公式 / 操作 | 需要观察的结果 |
|---|---|---|
| 动态范围 | `absmax = max(abs(x))` | 找到当前权重范围 |
| 缩放 | `scale = 127 / absmax` | 全零输入需要防止除零 |
| 量化 | `round(x * scale)` 后截断到 INT8 范围 | 存储为 `torch.int8` |
| 反量化 | `x_dequant = x_int8 / scale` | 得到近似浮点权重 |

![W8A16 对称量化机制图](../docs/public/02_PyTorch_Algorithms/25_w8a16_math_flow_cn.svg)

### Step 4：实现、测试与结果解读

题目区用两个机制动作复现 W8A16：先把浮点权重映射为 `INT8 + scale`，再在前向时恢复近似权重并与高精度激活相乘。测试不会要求记忆公式，而是检查全零保护、INT8 存储、反量化参考式和输入 dtype。

| TODO | 实现对象 | 机制责任 | 关键测试 |
|:---|:---|:---|:---|
| TODO 1 | `absmax_quantize` | 用动态范围生成 scale，并完成 round / clamp / INT8 存储 | 全零输入、scale、INT8 范围 |
| TODO 2 | `W8A16Linear.forward` | 将 `INT8 + scale` 恢复为与激活同 dtype 的近似权重，再执行线性层 | 输出形状、反量化参考式、FP16 路径 |
| 辅助方法 | `from_float` 与 buffer | 将量化状态写入模型，并随模型保存或迁移 | INT8 buffer 与权重存储账本 |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# 题目区只挖空两个机制：对称量化映射，以及反量化后的线性前向。
# 输入检查、buffer 注册和权重拷贝由骨架提供。

def absmax_quantize(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """将浮点权重映射为 INT8 张量和一个对称量化 scale。

    ``scale`` 把当前张量的绝对最大值映射到 127；全零输入仍需返回有限的
    scale。量化结果必须限制在 INT8 范围内。
    """
    if not x.is_floating_point():
        raise TypeError('x 必须是浮点张量')
    # TODO 1（对称量化）：由 absmax 计算 scale，并完成缩放、round、clamp 与 INT8 转换。
    # absmax = ???
    # scale = ???       # 全零张量应使用有限的非零 scale。
    # x_quant = ???     # 先缩放和 round，再限制到 [-128, 127] 并转换为 torch.int8。
    return x_quant, scale


class W8A16Linear(nn.Module):
    """保存 INT8 权重和 scale，并以高精度激活执行教学版 W8A16 前向。"""

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.register_buffer('weight_int8', torch.zeros((out_features, in_features), dtype=torch.int8))
        self.register_buffer('scale', torch.tensor(1.0))
        self.bias = nn.Parameter(torch.zeros(out_features))

    def from_float(self, linear_layer: nn.Linear):
        """量化浮点 Linear 的权重，并保留可训练 bias。"""
        with torch.no_grad():
            w_quant, scale = absmax_quantize(linear_layer.weight)
            self.weight_int8.copy_(w_quant)
            self.scale.copy_(scale)
            if linear_layer.bias is not None:
                self.bias.copy_(linear_layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """将量化权重恢复到激活 dtype，并执行一次线性前向。"""
        # TODO 2（反量化前向）：INT8 权重、scale、bias 都要与输入激活的 dtype 对齐。
        # w_fp = ???        # 将 self.weight_int8 转成 x.dtype。
        # w_dequant = ???   # 用 self.scale 恢复近似浮点权重。
        # out = ???         # 使用 F.linear(x, w_dequant, bias)。
        return out


In [ ]:
# 机制测试：分别检查量化边界、存储账本、反量化前向和 dtype 契约。
def test_absmax_quantization_contract():
    """验证 absmax、scale、全零输入和 INT8 边界。"""
    zero_q, zero_scale = absmax_quantize(torch.zeros(5))
    assert zero_q.dtype == torch.int8
    assert torch.count_nonzero(zero_q) == 0
    assert torch.isfinite(torch.as_tensor(zero_scale)).item()

    x_fp = torch.tensor([-0.8, 1.5, -3.0, 2.5, 0.0])
    x_q, scale = absmax_quantize(x_fp)
    assert torch.allclose(scale, torch.tensor(127.0 / 3.0))
    assert x_q[3].item() == 106


def test_w8a16_storage_contract():
    """验证权重确实以 INT8 buffer 保存，并能计算存储差异。"""
    fp_linear = nn.Linear(128, 64)
    q_linear = W8A16Linear(128, 64)
    q_linear.from_float(fp_linear)
    fp_bytes = fp_linear.weight.element_size() * fp_linear.weight.numel()
    q_bytes = q_linear.weight_int8.element_size() * q_linear.weight_int8.numel()
    assert q_linear.weight_int8.dtype == torch.int8
    assert q_bytes * (fp_linear.weight.element_size() // q_linear.weight_int8.element_size()) == fp_bytes


def test_dequantized_linear_contract():
    """验证量化层前向与显式反量化参考路径一致。"""
    fp_linear = nn.Linear(4, 3)
    with torch.no_grad():
        fp_linear.weight.copy_(torch.tensor([
            [1.0, -2.0, 3.0, -4.0],
            [0.5, 0.25, -0.75, 1.5],
            [-1.0, 0.0, 1.0, -2.0],
        ]))
        fp_linear.bias.copy_(torch.tensor([0.1, -0.2, 0.3]))
    q_linear = W8A16Linear(4, 3)
    q_linear.from_float(fp_linear)
    x = torch.tensor([[1.0, -1.0, 0.5, 2.0], [0.0, 1.0, -1.0, 3.0]])
    out = q_linear(x)
    restored_weight = q_linear.weight_int8.to(x.dtype) / q_linear.scale
    reference = F.linear(x, restored_weight, q_linear.bias)
    assert torch.allclose(out, reference, atol=1e-6)


def test_dtype_and_output_contract():
    """验证输出形状、有限值、余弦相似度和 FP16 输入路径。"""
    torch.manual_seed(42)
    fp_linear = nn.Linear(128, 64)
    q_linear = W8A16Linear(128, 64)
    q_linear.from_float(fp_linear)
    x_fp32 = torch.randn(2, 10, 128)
    out_fp = fp_linear(x_fp32)
    out_q = q_linear(x_fp32)
    cosine = F.cosine_similarity(out_fp.flatten(), out_q.flatten(), dim=0)
    assert out_q.shape == out_fp.shape
    assert torch.isfinite(out_q).all()
    assert cosine > 0.99

    x_half = x_fp32.to(torch.float16)
    q_half = q_linear(x_half)
    assert q_half.dtype == torch.float16
    assert q_half.shape == out_fp.shape


def test_w8a16_integration():
    """验证量化线性层能够作为普通线性层完成一次集成前向。"""
    torch.manual_seed(42)
    fp_linear = nn.Linear(8, 4)
    q_linear = W8A16Linear(8, 4)
    q_linear.from_float(fp_linear)
    x = torch.randn(2, 8)
    assert q_linear(x).shape == fp_linear(x).shape


def run_w8a16_tests():
    for test in (
        test_absmax_quantization_contract,
        test_w8a16_storage_contract,
        test_dequantized_linear_contract,
        test_dtype_and_output_contract,
        test_w8a16_integration,
    ):
        test()
    print('✅ W8A16 机制测试通过：量化、存储、反量化、dtype 与集成路径均已验证。')


run_w8a16_tests()


## 参考代码与解析

### 代码

In [ ]:
def absmax_quantize(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """将浮点权重映射为 INT8 张量和一个对称量化 scale。"""
    if not x.is_floating_point():
        raise TypeError('x 必须是浮点张量')
    # TODO 1：用动态范围建立对称映射，并避免全零输入除零。
    absmax = torch.max(torch.abs(x))
    safe_absmax = torch.where(absmax == 0, torch.ones_like(absmax), absmax)
    scale = 127.0 / safe_absmax
    x_quant = torch.clamp(torch.round(x * scale), -128, 127).to(torch.int8)
    return x_quant, scale


class W8A16Linear(nn.Module):
    """保存 INT8 权重和 scale，并以高精度激活执行教学版 W8A16 前向。"""

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.register_buffer('weight_int8', torch.zeros((out_features, in_features), dtype=torch.int8))
        self.register_buffer('scale', torch.tensor(1.0))
        self.bias = nn.Parameter(torch.zeros(out_features))

    def from_float(self, linear_layer: nn.Linear):
        """量化浮点 Linear 的权重，并保留可训练 bias。"""
        with torch.no_grad():
            w_quant, scale = absmax_quantize(linear_layer.weight)
            self.weight_int8.copy_(w_quant)
            self.scale.copy_(scale)
            if linear_layer.bias is not None:
                self.bias.copy_(linear_layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """将量化权重恢复到激活 dtype，并执行一次线性前向。"""
        # TODO 2：先对齐 dtype，再按照 ``INT8 / scale`` 反量化。
        w_fp = self.weight_int8.to(dtype=x.dtype)
        w_dequant = w_fp / self.scale.to(dtype=x.dtype)
        bias = self.bias.to(dtype=x.dtype)
        return F.linear(x, w_dequant, bias)


### 解析

**TODO 1：对称量化映射**

- `absmax` 描述当前权重的动态范围；`scale = 127 / absmax` 把该范围映射到 INT8 的正向上限。
- 全零张量没有有效动态范围，因此用 `1` 作为安全分母：量化结果仍全为零，scale 也保持有限。
- `round` 引入离散化误差，`clamp` 保证结果可存入 `torch.int8`。本题采用 `[-128, 127]` 的存储范围，而 scale 以对称的 127 为基准。

**TODO 2：反量化前向**

- `weight_int8.to(x.dtype) / scale` 恢复近似权重；激活 `x` 没有在本题中量化，这就是 A16 的含义。
- `F.linear` 的输入、权重和 bias 需要同 dtype，才能同时覆盖 FP32 与 FP16 输入路径。
- 测试将输出与显式反量化参考式比较，因此它验证的是表示与前向关系，而不是某个 GPU INT8 kernel 的速度。

**模型状态如何保存**

- `weight_int8` 与 `scale` 是 buffer：它们随 `state_dict()` 保存、随设备迁移，但不会被优化器更新。
- 存储测试只统计权重张量本身；scale 与 bias 仍有开销，所以它不能替代完整模型的显存账本。


### Step 5：可选 GPU 实验——测量 W8A16 的存储与反量化代价

![W8A16 GPU 机制实验流程](../docs/public/02_PyTorch_Algorithms/25_w8a16_gpu_mechanism_flow.svg)

实验在 FP16 baseline 和 W8A16 线性层之间做对照。它只测真实层的权重存储、前向耗时、峰值显存和输出误差；不等价于真实 INT8 kernel，也不是完整模型部署。

#### 5.1 环境、输入与固定条件

先运行 dry_run 检查环境，固定模型 revision、层尺寸、dtype、batch、序列长度、warmup 和重复次数。当前小节只比较真实模型层的 W8A16 存储与教学反量化路径；真实 INT8 kernel 和端到端部署收益由 67 节验证。

| 证据等级 | 本节可以说明什么 | 不应直接推出什么 |
|:---|:---|:---|
| environment_preflight | 当前环境、模型来源和配置可以开始测量 | 没有产生 GPU 性能或质量结论 |
| gpu_real_model_state_measurement | 真实模型层的权重字节数、教学反量化路径和固定输入下的相对变化 | 目标 GPU 是否使用真实 INT8 kernel、完整模型吞吐或服务质量 |
| real_backend_benchmark | 由 67 在固定 backend 和 workload 下验证加载、kernel、显存、延迟、吞吐和质量 | 不能外推到其他硬件、版本或 workload |

#### 5.2 执行实验并保存 JSON

在固定 workload 下分别测量 FP16 baseline 与 W8A16 candidate；代码默认只做 dry run，开启 GPU 模式后保存运行环境、指标、失败状态和 evidence level。


In [ ]:
# 5.1 只准备固定 workload；默认不下载模型、不启动 GPU 测量。
from pathlib import Path

RUN_MODE = 'dry_run'  # dry_run / real_gpu
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
PROMPT = 'Explain why lower precision can reduce weight storage.'
SEED = 42
IN_FEATURES = 4096
OUT_FEATURES = 4096
BATCH_SIZE = 1
SEQ_LEN = 128
DTYPE = torch.float16
WARMUP = 5
ITERS = 20
OUTPUT_PATH = Path('benchmarks/results/25_w8a16_gpu.json')


In [ ]:
import json
import platform
import time
# 5.2 读取 5.1 的固定配置并执行；请先运行上一个配置单元。

torch.manual_seed(SEED)
cuda_available = torch.cuda.is_available()
if RUN_MODE == 'real_gpu' and not cuda_available:
    raise RuntimeError('RUN_MODE=real_gpu 但 CUDA 不可用，请先完成 GPU 环境预检。')
device = torch.device('cuda' if RUN_MODE == 'real_gpu' else 'cpu')
runtime = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'cuda_available': cuda_available,
    'device': torch.cuda.get_device_name(0) if cuda_available else 'cpu',
}

def _sync():
    """确保 CUDA 异步操作完成后再读取计时或显存。"""
    if device.type == 'cuda':
        torch.cuda.synchronize()

def _measure(fn):
    """在固定 warmup / iters 下测量一个前向函数。"""
    for _ in range(WARMUP): fn()
    _sync()
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    for _ in range(ITERS): fn()
    _sync()
    elapsed = (time.perf_counter() - start) * 1000 / ITERS
    peak = torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None
    return {'latency_ms': round(elapsed, 4), 'peak_memory_mb': None if peak is None else round(peak, 2)}

evidence_level = 'environment_preflight' if RUN_MODE == 'dry_run' else 'gpu_real_model_state_measurement'
result = {'stage': evidence_level, 'run_mode': RUN_MODE, 'runtime': runtime, 'config': {
    'in_features': IN_FEATURES, 'out_features': OUT_FEATURES, 'batch_size': BATCH_SIZE,
    'seq_len': SEQ_LEN, 'dtype': str(DTYPE), 'model_id': MODEL_ID, 'warmup': WARMUP, 'iters': ITERS, 'seed': SEED,
}, 'workload': {'model_id': MODEL_ID, 'layer_scope': 'one q_proj-like linear layer', 'batch_size': BATCH_SIZE, 'seq_len': SEQ_LEN},
   'json_path': str(OUTPUT_PATH), 'evidence_level': evidence_level, 'baseline': 'FP16 layer in memory',
   'candidate': 'W8A16 layer in memory', 'artifact_path': None, 'failure': None}
if RUN_MODE == 'dry_run':
    result['decision'] = {'decision': 'ready_to_measure', 'reason': '仅完成环境与配置检查，尚未运行 GPU 机制测量。'}
else:
    # real_gpu 分支加载真实模型；CPU / dry_run 不下载模型，避免把预检变成训练任务。
    if RUN_MODE == 'real_gpu':
        from transformers import AutoModelForCausalLM, AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(device).eval()
        input_ids = tokenizer(PROMPT, return_tensors='pt').input_ids.to(device)
        with torch.no_grad(): x = model.model.embed_tokens(input_ids).to(DTYPE)
        source = model.model.layers[0].self_attn.q_proj
        IN_FEATURES, OUT_FEATURES = source.in_features, source.out_features
        fp = nn.Linear(IN_FEATURES, OUT_FEATURES, bias=source.bias is not None, device=device, dtype=DTYPE)
        with torch.no_grad():
            fp.weight.copy_(source.weight.to(DTYPE))
            if source.bias is not None: fp.bias.copy_(source.bias.to(DTYPE))
        del model, source
        if device.type == 'cuda': torch.cuda.empty_cache()
    else:
        # cpu 模式只运行小型确定性张量，真实模型状态验证留给 real_gpu。
        weight = torch.randn(OUT_FEATURES, IN_FEATURES, device=device, dtype=DTYPE)
        x = torch.randn(BATCH_SIZE, SEQ_LEN, IN_FEATURES, device=device, dtype=DTYPE)
        fp = nn.Linear(IN_FEATURES, OUT_FEATURES, bias=True, device=device, dtype=DTYPE)
        with torch.no_grad(): fp.weight.copy_(weight); fp.bias.zero_()
    result['config'].update({'in_features': IN_FEATURES, 'out_features': OUT_FEATURES,
                           'actual_input_shape': list(x.shape), 'weight_source': 'real_model' if RUN_MODE == 'real_gpu' else 'synthetic_cpu'})
    q = W8A16Linear(IN_FEATURES, OUT_FEATURES).to(device)
    q.from_float(fp.float())
    baseline = _measure(lambda: fp(x))
    candidate = _measure(lambda: q(x))
    with torch.no_grad():
        ref, approx = fp(x), q(x)
        mse = torch.mean((ref.float() - approx.float()) ** 2).item()
    result.update({'baseline': baseline, 'candidate': candidate, 'metrics': {
        'weight_fp_bytes': int(fp.weight.numel() * fp.weight.element_size()),
        'weight_int8_bytes': int(q.weight_int8.numel() * q.weight_int8.element_size()),
        'output_mse': round(mse, 8), 'output_shape': list(approx.shape),
    }, 'decision': {'decision': 'measure', 'reason': '仅验证 W8A16 GPU 机制；不代表真实 INT8 kernel 加速。'}})
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(result, ensure_ascii=False, indent=2))

#### 5.3 读取结果与解释证据

将 RUN_MODE 切换为 real_gpu 并完成 5.2 后，先读取保存的 baseline / candidate、配置、runtime、失败状态和结果 JSON。

**成熟库探针（可选）**：如果环境已安装兼容版本的 `torchao`，可以在同一固定 workload 下验证 PyTorch 原生 INT8 weight-only 路径。该探针不替代上面的手写实现，也不把单层结果解释为完整模型收益。


In [ ]:
# 5.3 只读取 5.2 保存的主实验结果；不重新启动前向测量。
if OUTPUT_PATH.exists():
    saved = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
    print({key: saved.get(key) for key in ('workload', 'config', 'metrics', 'failure', 'evidence_level', 'decision')})
else:
    print(f'等待 GPU W8A16 结果：{OUTPUT_PATH}')


In [ ]:
RUN_TORCHAO_PROBE = False  # 默认关闭；需要成熟库验证时改为 True
TORCHAO_OUTPUT = Path('benchmarks/results/25_w8a16_torchao_probe.json')

if not RUN_TORCHAO_PROBE:
    print('torchao probe skipped; set RUN_TORCHAO_PROBE=True on a compatible CUDA environment.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('RUN_TORCHAO_PROBE=True requires CUDA.')
    try:
        from torchao.quantization import Int8WeightOnlyConfig, quantize_
    except ImportError as exc:
        raise ImportError('请安装与当前 PyTorch 兼容的 torchao，再运行成熟库探针。') from exc
    torch.manual_seed(SEED)
    probe = nn.Linear(IN_FEATURES, OUT_FEATURES, device='cuda', dtype=DTYPE).eval()
    probe_input = torch.randn(BATCH_SIZE, SEQ_LEN, IN_FEATURES, device='cuda', dtype=DTYPE)
    quantize_(probe, Int8WeightOnlyConfig())
    torch.cuda.synchronize()
    with torch.inference_mode():
        probe_output = probe(probe_input)
    torch.cuda.synchronize()
    probe_result = {
        'json_path': str(TORCHAO_OUTPUT),
        'workload': {'layer_scope': 'one Linear layer', 'batch_size': BATCH_SIZE, 'seq_len': SEQ_LEN},
        'baseline': 'FP16 Linear in memory', 'candidate': 'torchao Int8WeightOnlyConfig',
        'library': 'torchao',
        'config': 'Int8WeightOnlyConfig',
        'input_shape': list(probe_input.shape),
        'output_shape': list(probe_output.shape),
        'device': torch.cuda.get_device_name(0),
        'torchao_version': getattr(__import__('torchao'), '__version__', 'unknown'),
        'evidence_level': 'mature_library_gpu_probe',
        'decision': 'api_path_executed_not_full_backend_benchmark',
        'artifact_path': None,
        'failure': None,
    }
    TORCHAO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    TORCHAO_OUTPUT.write_text(json.dumps(probe_result, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(probe_result, ensure_ascii=False, indent=2))


#### 5.4 结果记录表

读取 JSON 后，先核对实际环境和固定配置，再比较 FP16 baseline 与 W8A16 的权重字节数、前向耗时、峰值显存和输出误差。权重字节数只描述本层权重存储，不等于完整模型显存；真实 INT8 kernel 和部署收益转到 67 验证。复测时如果环境、模型或 workload 不一致，应在 failure 中记录原因。

| role | baseline / candidate | artifact | runtime/config | dtype | batch / seq_len | 权重存储 | forward latency | peak memory | output MSE | failure | evidence level | decision |
|---|---|---|---|---|---|---:|---:|---:|---:|---|---|---|
| reference | baseline | FP16 model layer in memory / JSON path |  |  |  |  |  |  |  |  | gpu_real_model_state_measurement |  |
| quantized | candidate | W8A16 layer in memory / JSON path |  |  |  |  |  |  |  |  | gpu_real_model_state_measurement | accept / tune / reject |

## 相关阅读

完成 W8A16 的最小实现后，可以继续阅读量化校准方法和真实部署 backend。
- [SmoothQuant 原论文](https://arxiv.org/abs/2211.10438)
- [PyTorch 量化文档](https://pytorch.org/docs/stable/quantization.html)
- [26. QLoRA and 4-bit Quantization | QLoRA 与 4-bit 量化](./26_QLoRA_and_4bit_Quantization.ipynb)
- [40. GPTQ and AWQ Weight Quantization | GPTQ 与 AWQ 权重量化](./40_GPTQ_and_AWQ_Weight_Quantization.ipynb)
- [67. Quantized Inference and Deployment | 量化推理与部署](./67_Quantized_Inference_and_Deployment.ipynb)
